# Lab 04 — Troubleshooting and Performance Tuning

**Exam domain:** Troubleshooting and Tuning Apache Spark DataFrame API Applications (10% ≈ 4–5 questions)

**Time:** ~75 minutes

Small domain, high value: these questions are usually "given this symptom, what do you change?" — and
they are easy marks once you know the four levers (partitioning, broadcast, AQE, caching) and the
vocabulary of the Spark UI.

In [ ]:
try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("lab04-tuning").master("local[4]").getOrCreate()

from pyspark.sql import functions as F
import time

def timed(label, fn):
    t = time.time(); r = fn(); print(f"{label:44} {time.time()-t:6.2f}s"); return r

## 1. Data skew — spot it, then fix it

**Symptom in the Spark UI:** in a stage's *Summary Metrics*, the **Max** duration / shuffle-read is many
times the **Median**. 199 tasks finish in 2 s and one runs for 4 minutes. Often accompanied by
**spill (memory)** and **spill (disk)** on that one task, and eventually an executor OOM.

In [ ]:
# Build a deliberately skewed key distribution: 90% of rows share one key
n = 2_000_000
skewed = (spark.range(n)
          .withColumn("k", F.when(F.col("id") % 10 != 0, F.lit(0))
                            .otherwise((F.col("id") % 500).cast("long")))
          .withColumn("v", F.rand()))

dim = spark.range(0, 500).withColumnRenamed("id", "k").withColumn("label", F.concat(F.lit("k"), F.col("k")))

skewed.groupBy("k").count().orderBy(F.desc("count")).show(5)

In [ ]:
# See the imbalance directly
from pyspark.sql.functions import spark_partition_id
(skewed.repartition(16, "k")
       .withColumn("pid", spark_partition_id())
       .groupBy("pid").count().orderBy(F.desc("count")).show(16))

### The fixes, in the order you should try them

1. **Broadcast the small side** — if the other table fits, the skewed key never gets shuffled at all.
   This solves most real skew.
2. **Turn on AQE skew join** (`spark.sql.adaptive.skewJoin.enabled`, on by default in 3.5). At runtime
   Spark detects a partition that is both bigger than `skewedPartitionFactor` × the median **and**
   bigger than `skewedPartitionThresholdInBytes` (default 256 MB), and splits it into sub-partitions.
3. **Salting** — add a random suffix to the hot key on the big side and explode the small side to match.
   Manual, ugly, but works when AQE can't (e.g. skew in a `groupBy`).
4. **Filter the hot key out**, process it separately, union the results.

In [ ]:
# Salting a skewed join, the pattern to be able to recognise
SALT = 8
big_salted = skewed.withColumn("salt", (F.rand() * SALT).cast("int"))
dim_salted = (dim.withColumn("salt", F.explode(F.array([F.lit(i) for i in range(SALT)]))))

salted_join = big_salted.join(dim_salted, on=["k", "salt"], how="inner")
print("salted join rows:", salted_join.count())
salted_join.explain("formatted")

## 2. Partition tuning

| Symptom | Likely cause | Fix |
|---|---|---|
| Thousands of tiny output files | too many shuffle partitions | `coalesce(n)` before write; AQE coalesce |
| A few huge tasks, lots of spill | too few partitions | raise `spark.sql.shuffle.partitions` / `repartition(n)` |
| Only 1 task on a big file | non-splittable compression (gzip) or a single JDBC read | change format; add JDBC partitioning options |
| Long GC pauses, executor lost | partitions too large for executor memory | more partitions, less caching, bigger executors |

Rules of thumb the exam likes:
- Aim for partitions of roughly **100–200 MB**.
- Target **2–4 tasks per CPU core** so stragglers even out.
- `spark.sql.shuffle.partitions` default is **200** — nearly always wrong for your data size, in one
  direction or the other.

In [ ]:
for parts in [4, 32, 400]:
    spark.conf.set("spark.sql.shuffle.partitions", parts)
    timed(f"groupBy with shuffle.partitions={parts}",
          lambda: skewed.groupBy("k").agg(F.avg("v")).count())
spark.conf.set("spark.sql.shuffle.partitions", 200)

## 3. Adaptive Query Execution (AQE)

AQE re-optimises the plan **at runtime**, using real statistics collected from completed shuffle stages —
something Catalyst cannot do at plan time because it only has estimates. Enabled by default since
Spark 3.2.

Three things it does (know all three by name):

| Feature | Config | What it does |
|---|---|---|
| **Coalescing shuffle partitions** | `spark.sql.adaptive.coalescePartitions.enabled` | merges lots of tiny post-shuffle partitions into fewer right-sized ones — fixes the "200 tiny files" problem automatically |
| **Switching join strategies** | `spark.sql.adaptive.autoBroadcastJoinThreshold` | converts a planned sort-merge join into a broadcast hash join once it sees the real size of a side |
| **Optimizing skew joins** | `spark.sql.adaptive.skewJoin.enabled` | splits oversized partitions into several tasks |

In the physical plan AQE shows up as `AdaptiveSparkPlan isFinalPlan=false` before execution and
`isFinalPlan=true` after, plus `CustomShuffleReader`/`AQEShuffleRead coalesced` nodes.

AQE **cannot** help when there is no shuffle at all, and it does not change the *logical* plan.

In [ ]:
for k, v in [("spark.sql.adaptive.enabled", None),
             ("spark.sql.adaptive.coalescePartitions.enabled", None),
             ("spark.sql.adaptive.coalescePartitions.minPartitionSize", None),
             ("spark.sql.adaptive.advisoryPartitionSizeInBytes", None),
             ("spark.sql.adaptive.skewJoin.enabled", None),
             ("spark.sql.adaptive.skewJoin.skewedPartitionFactor", None),
             ("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", None),
             ("spark.sql.adaptive.autoBroadcastJoinThreshold", None)]:
    try:    print(f"{k:62} = {spark.conf.get(k)}")
    except Exception: print(f"{k:62} = <unset>")

In [ ]:
# AQE on vs off, on a query that produces many small post-shuffle partitions
q = lambda: skewed.groupBy("k").agg(F.count("*")).write.format("noop").mode("overwrite").save()

spark.conf.set("spark.sql.adaptive.enabled", False); timed("AQE off", q)
spark.conf.set("spark.sql.adaptive.enabled", True);  timed("AQE on ", q)

skewed.groupBy("k").agg(F.count("*")).explain("formatted")   # look for AdaptiveSparkPlan

> **`.format("noop")`** is the benchmarking sink: it executes the whole plan but writes nothing. Use it
> instead of `.count()` when you want to measure the real write path without I/O.

## 4. Reading a physical plan

Run `df.explain("formatted")` and scan for these, in rough order of "how worried should I be":

| Node | Meaning |
|---|---|
| `FileScan parquet ... PushedFilters: [...]` | good — filter went to the reader |
| `PartitionFilters: [...]` | good — partition pruning happened |
| `Exchange hashpartitioning(...)` | a shuffle. Count them. |
| `SortMergeJoin` | both sides shuffled + sorted. Can it be a broadcast instead? |
| `BroadcastHashJoin` + `BroadcastExchange` | the good join |
| `BroadcastNestedLoopJoin` | ⚠️ usually means a non-equi join condition — often accidental, often catastrophic |
| `CartesianProduct` | ⚠️ you forgot a join condition |
| `*(3) Project` | whole-stage codegen, group 3 — fine |
| `ColumnarToRow` | boundary between vectorised read and row processing |
| `Scan ExistingRDD` | you built a DataFrame from Python objects — no pushdown possible |

In [ ]:
# Compare a filter BEFORE vs AFTER a join — and see Catalyst move it for you
import os, shutil
p = "/tmp/tuning_demo"
shutil.rmtree(p, ignore_errors=True)
skewed.limit(200_000).write.mode("overwrite").parquet(p)
src = spark.read.parquet(p)

late  = src.join(dim, "k").filter(F.col("k") == 3)
early = src.filter(F.col("k") == 3).join(dim, "k")

print("### filter written AFTER the join:"); late.explain("formatted")
print("### filter written BEFORE the join:"); early.explain("formatted")
# Catalyst pushes the predicate down in both cases -> nearly identical physical plans.
# Predicate pushdown is one of the things a Python UDF blocks.

In [ ]:
# What a UDF does to the plan: the filter can no longer be pushed into the scan
@F.udf("boolean")
def is_three(k):
    return k == 3

src.filter(is_three("k")).explain("formatted")   # PushedFilters is empty; note BatchEvalPython

## 5. Logging and monitoring

**Where to look, in order:**

1. **Spark UI → Jobs** — which job is slow, how many stages.
2. **Stages → Summary Metrics** — Min/25th/Median/75th/Max for Duration, GC Time, Shuffle Read/Write,
   Spill. Max ≫ Median ⇒ skew.
3. **SQL/DataFrame tab** — the query graph with row counts and bytes per node. The best place to see
   "this join produced 4 billion rows".
4. **Executors tab** — cores in use, memory used, **GC time** as a fraction of task time
   (>10% is a red flag), failed tasks, shuffle read/write per executor.
5. **Storage tab** — is your cached DataFrame actually 100% cached, or evicted?
6. **Driver log** — your `print`s, the stack trace of the *originating* exception, `OutOfMemoryError`.
7. **Executor logs (stderr)** — the *real* task failure, `ExecutorLostFailure`, container-killed-by-YARN.

**Symptom → cause cheat sheet**

| Message / symptom | Usual cause |
|---|---|
| `OutOfMemoryError` on the **driver** | `collect()` / `toPandas()` on a big DataFrame, or broadcasting something too large |
| `ExecutorLostFailure` / container killed, exceeding memory limits | partitions too large, skew, too much cached data |
| `Job aborted due to stage failure: Task ... FetchFailedException` | shuffle service / executor died mid-shuffle — usually a downstream symptom of OOM |
| Lots of **spill (disk)** | partitions bigger than executor memory ⇒ more partitions |
| `AnalysisException` | resolution-time error: bad column name, ambiguous reference, missing table. Happens **before** any job runs |
| `Py4JJavaError` wrapping the above | you're seeing the JVM exception through Python |
| Job is fast but the write takes forever | too many output partitions ⇒ too many small files |
| Query returns nulls where you expected values | UDF returned the wrong type, or a date format didn't parse |

## 6. Executor memory model (worth 5 minutes)

Executor JVM heap ≈ `spark.executor.memory`, split by:

- **Reserved** — 300 MB, fixed.
- **Unified region** = (heap − reserved) × `spark.memory.fraction` (**0.6**), split into
  - **Execution memory** — shuffles, joins, sorts, aggregations.
  - **Storage memory** — cached partitions and broadcast variables.
  The boundary is `spark.memory.storageFraction` (**0.5**) but it is **soft**: execution can evict
  storage; storage can **never** evict execution.
- **User memory** — the remaining 40%: your data structures, UDF state.

Plus **off-heap overhead** (`spark.executor.memoryOverhead`, default max(384 MB, 10%)) for Python
workers, network buffers and the JVM itself. **PySpark UDFs run in separate Python processes whose
memory comes out of the overhead, not the heap** — which is why heavy PySpark UDF jobs die with
"container killed, exceeding physical memory" while the heap looks fine.

### ✍️ Exercises

**4.1** A job joins a 500 GB fact table to a 50 MB dimension. The plan shows `SortMergeJoin`. Give two
ways to make it a broadcast join and one reason it might still be a bad idea.

**4.2** Your write produces 200 files averaging 3 MB. What's happening and what are two fixes?

**4.3** A `groupBy("user_id")` on clickstream data has one task running 40× longer than the rest.
AQE is on. Why might AQE not be helping, and what would you do?

<details><summary>Answers</summary>

**4.1** `F.broadcast(dim)` or the SQL hint `/*+ BROADCAST(dim) */`; alternatively raise
`spark.sql.autoBroadcastJoinThreshold` above 50 MB. Bad idea if the driver is small (the table is
collected to the driver first) or if there are hundreds of executors, since the table is shipped to each.

**4.2** 200 = the default `spark.sql.shuffle.partitions`, and the last stage was a shuffle. Fix with
`coalesce(n)` (or `repartition(n)`) before writing, or rely on AQE's `coalescePartitions`, or set
`spark.sql.adaptive.advisoryPartitionSizeInBytes` to your target file size.

**4.3** AQE's skew handling applies to **joins**, not to aggregations. Fixes: two-stage aggregation with
salting (aggregate by `user_id` + salt, then re-aggregate), pre-filtering the bot/hot user, or switching
to an approximate aggregate.
</details>

## 7. Wrap-up checklist

- [ ] I can name AQE's three optimisations and the config keys.
- [ ] I know the default broadcast threshold (10 MB) and how to disable it (-1).
- [ ] I can describe salting in two sentences.
- [ ] I know Max ≫ Median in Summary Metrics means skew.
- [ ] I know which side of the memory boundary can evict the other (execution evicts storage, never the reverse).
- [ ] I know why PySpark UDF jobs die on *overhead* memory rather than heap.
- [ ] I can tell an `AnalysisException` (before execution) from a task failure (during execution).
- [ ] I know `.format("noop")` for benchmarking.